# Tracer Core Utilities

The `core.py` module provides the internal run-construction and lifecycle engine shared by LangChain tracer implementations.

It creates `Run` objects for chat models, language models, chains, tools, and retrievers; records token, retry, completion, and error events; maintains parent-child trace relationships; and exposes protected hooks used by synchronous and asynchronous tracer base classes.

## Re-exports

1. `TracerException`: Re-exports the tracer exception class from `langchain_core.exceptions`.

## Type Aliases

1. `SCHEMA_FORMAT_TYPE`: Represents the primary supported tracer schema formats.
   * **Definition:**
     ```python
     SCHEMA_FORMAT_TYPE = Literal[
         "original",
         "streaming_events"
     ]
     ```

# _TracerCore

`_TracerCore` is the internal abstract base class that contains reusable tracing logic.

Although its name is private, it defines the protected implementation contract used by the public synchronous and asynchronous tracer base classes.

## Bases

- `ABC`

## Attributes

1. `log_missing_parent`: Controls whether a debug message is logged when a run references a parent that is not present in the trace-order map.
   * **Type:**
     ```python
     log_missing_parent: bool = True
     ```

2. `run_map`: Maps string run identifiers to active `Run` objects.

   Entries are expected to be cleared when their runs end.

   * **Type:**
     ```python
     run_map: dict[
         str,
         Run
     ]
     ```

3. `order_map`: Maps run identifiers to their trace identifier and dotted-order string.

   This map is retained for trace ordering and is cleared when the tracer is garbage-collected.

   * **Type:**
     ```python
     order_map: dict[
         UUID,
         tuple[
             UUID,
             str
         ]
     ]
     ```

4. `_external_run_ids`: Stores active-child reference counts for externally inserted parent runs.

   These parent runs may exist in `run_map` so that child runs can resolve them, even though their lifecycle is not managed directly by the tracer.

   * **Type:**
     ```python
     _external_run_ids: dict[
         str,
         int
     ]
     ```

### Methods

1. `__init__`: Initializes the tracer's schema format and shared run-tracking maps.

   The `"original"` format preserves dictionary inputs and outputs and wraps non-dictionary values. The `"streaming_events"` format stores values under a single `"input"` or `"output"` key. The internal `"original+chat"` format behaves like `"original"` while allowing direct chat-model run creation.

   * **Syntax:**
     ```python
     __init__(
         self,
         *,
         _schema_format: Literal[
             "original",
             "streaming_events",
             "original+chat"
         ] = "original", # Internal input and output schema format
         run_map: dict[
             str,
             Run
         ] | None = None, # Optional shared map of active runs
         order_map: dict[
             UUID,
             tuple[
                 UUID,
                 str
             ]
         ] | None = None, # Optional shared trace-order map
         _external_run_ids: dict[
             str,
             int
         ] | None = None, # Optional external-parent reference counts
         **kwargs: Any # Arguments passed to the superclass
     ) -> None
     ```

2. `_persist_run`: Persists a completed root run.

   Subclasses must implement this protected abstract method. An asynchronous implementation may return a coroutine.

   * **Syntax:**
     ```python
     @abstractmethod
     _persist_run(
         self,
         run: Run # Run to persist
     ) -> Coroutine[
         Any,
         Any,
         None
     ] | None
     ```

3. `_add_child_run`: Appends a child run to its parent run's `child_runs` collection.
   * **Syntax:**
     ```python
     @staticmethod
     _add_child_run(
         parent_run: Run, # Parent run
         child_run: Run # Child run to attach
     ) -> None
     ```

4. `_get_stacktrace`: Converts an exception and its traceback into a stored error string.

   When traceback formatting itself fails, the exception's `repr` value is returned.

   * **Syntax:**
     ```python
     @staticmethod
     _get_stacktrace(
         error: BaseException # Exception to format
     ) -> str
     ```

5. `_start_trace`: Registers a run and calculates its trace ordering information.

   A root run receives its own identifier as `trace_id`. A child run inherits its parent's `trace_id` and extends the parent's dotted order with its start timestamp and identifier.

   When the requested parent is unavailable, the run is treated as a root run. When the parent is externally inserted, its active-child reference count is incremented.

   * **Syntax:**
     ```python
     _start_trace(
         self,
         run: Run # Run to register
     ) -> Coroutine[
         Any,
         Any,
         None
     ] | None
     ```

6. `_get_run`: Retrieves an active run and optionally validates its run type.

   A `TracerException` is raised when the run identifier is unknown or when its type does not match the required type or set of types.

   * **Syntax:**
     ```python
     _get_run(
         self,
         run_id: UUID, # Run identifier to retrieve
         run_type: str
         | set[str]
         | None = None # Optional accepted run type or types
     ) -> Run
     ```

7. `_create_chat_model_run`: Creates a chat-model `Run`.

   Message objects are serialized with `dumpd`, metadata is added to the run's extra values, and an initial `"start"` event is recorded.

   Direct chat-model tracing is supported only by the `"streaming_events"` and `"original+chat"` formats. A `NotImplementedError` is raised for the `"original"` format so older tracers can fall back to LLM callbacks.

   * **Syntax:**
     ```python
     _create_chat_model_run(
         self,
         serialized: dict[
             str,
             Any
         ], # Serialized chat model
         messages: list[
             list[BaseMessage]
         ], # Batches of input messages
         run_id: UUID, # Unique run identifier
         tags: list[str] | None = None, # Optional run tags
         parent_run_id: UUID | None = None, # Optional parent-run identifier
         metadata: dict[
             str,
             Any
         ] | None = None, # Optional run metadata
         name: str | None = None, # Optional run name
         **kwargs: Any # Additional run data
     ) -> Run
     ```

8. `_create_llm_run`: Creates a text-LLM `Run`.

   Prompts are stored under the `"prompts"` input key, metadata is added to the extra values, and a `"start"` event is recorded.

   * **Syntax:**
     ```python
     _create_llm_run(
         self,
         serialized: dict[
             str,
             Any
         ], # Serialized language model
         prompts: list[str], # Input prompts
         run_id: UUID, # Unique run identifier
         tags: list[str] | None = None, # Optional run tags
         parent_run_id: UUID | None = None, # Optional parent-run identifier
         metadata: dict[
             str,
             Any
         ] | None = None, # Optional run metadata
         name: str | None = None, # Optional run name
         **kwargs: Any # Additional run data
     ) -> Run
     ```

9. `_llm_run_with_token_event`: Appends a `"new_token"` event to an active LLM or chat-model run.

   The event contains the token or content blocks and may also contain the associated generation chunk.

   * **Syntax:**
     ```python
     _llm_run_with_token_event(
         self,
         token: str
         | list[
             str
             | dict[
                 str,
                 Any
             ]
         ], # Token or structured content blocks
         run_id: UUID, # Run receiving the event
         chunk: GenerationChunk
         | ChatGenerationChunk
         | None = None, # Optional generation chunk
         parent_run_id: UUID | None = None # Compatibility parent-run identifier
     ) -> Run
     ```

10. `_llm_run_with_retry_event`: Appends a `"retry"` event to an active run.

    The event records the attempt number, total idle time, and the retry outcome. Failed outcomes include the exception text and exception type, while successful outcomes include the string representation of the result.

    * **Syntax:**
      ```python
      _llm_run_with_retry_event(
          self,
          retry_state: RetryCallState, # Tenacity retry state
          run_id: UUID # Run receiving the retry event
      ) -> Run
      ```

11. `_complete_llm_run`: Completes an LLM or chat-model run successfully.

    The response is merged into the run output unless `__omit_auto_outputs` is enabled. Chat messages inside generations are serialized, an `"end"` event and end time are recorded, and the number of generated tool calls is added to the run's extra values when greater than zero.

    * **Syntax:**
      ```python
      _complete_llm_run(
          self,
          response: LLMResult, # Completed model result
          run_id: UUID # Run to complete
      ) -> Run
      ```

12. `_errored_llm_run`: Completes an LLM or chat-model run with an error.

    The formatted traceback is stored on the run. When a partial response is supplied, its output is processed in the same way as a successful response. An `"error"` event and end time are recorded.

    * **Syntax:**
      ```python
      _errored_llm_run(
          self,
          error: BaseException, # Error raised during model execution
          run_id: UUID, # Run to complete
          response: LLMResult | None = None # Optional partial model response
      ) -> Run
      ```

13. `_create_chain_run`: Creates a chain or custom chain-like `Run`.

    Input formatting depends on the configured schema format. The run includes a `"start"` event, optional metadata, tags, and an empty child-run collection.

    * **Syntax:**
      ```python
      _create_chain_run(
          self,
          serialized: dict[
              str,
              Any
          ], # Serialized chain
          inputs: dict[
              str,
              Any
          ], # Chain inputs
          run_id: UUID, # Unique run identifier
          tags: list[str] | None = None, # Optional run tags
          parent_run_id: UUID | None = None, # Optional parent-run identifier
          metadata: dict[
              str,
              Any
          ] | None = None, # Optional run metadata
          run_type: str | None = None, # Optional custom run type
          name: str | None = None, # Optional run name
          **kwargs: Any # Additional run data
      ) -> Run
      ```

14. `_get_chain_inputs`: Converts a chain input into the structure required by the configured schema format.

    The original formats preserve dictionaries and wrap other values under `"input"`. The streaming-events format always wraps the value under `"input"`. A `ValueError` is raised for an unsupported format.

    * **Syntax:**
      ```python
      _get_chain_inputs(
          self,
          inputs: Any # Chain input to format
      ) -> Any
      ```

15. `_get_chain_outputs`: Converts a chain output into the structure required by the configured schema format.

    The original formats preserve dictionaries and wrap other values under `"output"`. The streaming-events format always wraps the value under `"output"`. A `ValueError` is raised for an unsupported format.

    * **Syntax:**
      ```python
      _get_chain_outputs(
          self,
          outputs: Any # Chain output to format
      ) -> Any
      ```

16. `_complete_chain_run`: Completes a chain run successfully.

    Formatted outputs are merged unless automatic outputs are disabled. The end time and an `"end"` event are recorded, and optional late inputs replace the stored input values.

    * **Syntax:**
      ```python
      _complete_chain_run(
          self,
          outputs: dict[
              str,
              Any
          ], # Chain outputs
          run_id: UUID, # Run to complete
          inputs: dict[
              str,
              Any
          ] | None = None # Optional late chain inputs
      ) -> Run
      ```

17. `_errored_chain_run`: Completes a chain run with an error.

    The formatted traceback, end time, and `"error"` event are stored. Optional late inputs replace the stored input values.

    * **Syntax:**
      ```python
      _errored_chain_run(
          self,
          error: BaseException, # Error raised during chain execution
          inputs: dict[
              str,
              Any
          ] | None, # Optional late chain inputs
          run_id: UUID # Run to complete
      ) -> Run
      ```

18. `_create_tool_run`: Creates a tool `Run`.

    In the original formats, structured dictionary input is preserved and other input is stored under `"input"`. In the streaming-events format, the structured input is wrapped under `"input"`. An `AssertionError` is raised for an unsupported schema format.

    * **Syntax:**
      ```python
      _create_tool_run(
          self,
          serialized: dict[
              str,
              Any
          ], # Serialized tool
          input_str: str, # String representation of the tool input
          run_id: UUID, # Unique run identifier
          tags: list[str] | None = None, # Optional run tags
          parent_run_id: UUID | None = None, # Optional parent-run identifier
          metadata: dict[
              str,
              Any
          ] | None = None, # Optional run metadata
          name: str | None = None, # Optional run name
          inputs: dict[
              str,
              Any
          ] | None = None, # Optional structured tool input
          **kwargs: Any # Additional run data
      ) -> Run
      ```

19. `_complete_tool_run`: Completes a tool run successfully.

    The tool output is stored under `"output"` unless automatic output insertion is disabled. The end time and an `"end"` event are recorded.

    * **Syntax:**
      ```python
      _complete_tool_run(
          self,
          output: dict[
              str,
              Any
          ], # Tool output
          run_id: UUID # Run to complete
      ) -> Run
      ```

20. `_errored_tool_run`: Completes a tool run with an error.

    The formatted traceback, end time, and `"error"` event are recorded.

    * **Syntax:**
      ```python
      _errored_tool_run(
          self,
          error: BaseException, # Error raised during tool execution
          run_id: UUID # Run to complete
      ) -> Run
      ```

21. `_create_retrieval_run`: Creates a retriever `Run`.

    The query is stored under `"query"`. The default run name is `"Retriever"`, and the run begins with a `"start"` event and an empty child-run collection.

    * **Syntax:**
      ```python
      _create_retrieval_run(
          self,
          serialized: dict[
              str,
              Any
          ], # Serialized retriever
          query: str, # Retrieval query
          run_id: UUID, # Unique run identifier
          parent_run_id: UUID | None = None, # Optional parent-run identifier
          tags: list[str] | None = None, # Optional run tags
          metadata: dict[
              str,
              Any
          ] | None = None, # Optional run metadata
          name: str | None = None, # Optional run name
          **kwargs: Any # Additional run data
      ) -> Run
      ```

22. `_complete_retrieval_run`: Completes a retriever run successfully.

    Retrieved documents are stored under `"documents"` unless automatic output insertion is disabled. The end time and an `"end"` event are recorded.

    * **Syntax:**
      ```python
      _complete_retrieval_run(
          self,
          documents: Sequence[Document], # Retrieved documents
          run_id: UUID # Run to complete
      ) -> Run
      ```

23. `_errored_retrieval_run`: Completes a retriever run with an error.

    The formatted traceback, end time, and `"error"` event are recorded.

    * **Syntax:**
      ```python
      _errored_retrieval_run(
          self,
          error: BaseException, # Error raised during retrieval
          run_id: UUID # Run to complete
      ) -> Run
      ```

24. `__deepcopy__`: Returns the current tracer instance instead of creating a deep copy.
    * **Syntax:**
      ```python
      __deepcopy__(
          self,
          memo: dict[
              int,
              Any
          ] | None = None # Optional copy memo
      ) -> _TracerCore
      ```

25. `__copy__`: Returns the current tracer instance instead of creating a shallow copy.
    * **Syntax:**
      ```python
      __copy__(
          self
      ) -> _TracerCore
      ```

26. `_end_trace`: Provides a protected hook for trace-finalization bookkeeping.

    The core implementation performs no action and returns `None`. Subclasses may override it.

    * **Syntax:**
      ```python
      _end_trace(
          self,
          run: Run # Run whose trace is ending
      ) -> Coroutine[
          Any,
          Any,
          None
      ] | None
      ```

27. `_on_run_create`: Provides a protected hook called when a run is created.

    The core implementation performs no action.

    * **Syntax:**
      ```python
      _on_run_create(
          self,
          run: Run # Newly created run
      ) -> Coroutine[
          Any,
          Any,
          None
      ] | None
      ```

28. `_on_run_update`: Provides a protected hook called when a run is updated.

    The core implementation performs no action.

    * **Syntax:**
      ```python
      _on_run_update(
          self,
          run: Run # Updated run
      ) -> Coroutine[
          Any,
          Any,
          None
      ] | None
      ```

29. `_on_llm_start`: Provides a protected hook called when an LLM run starts.

    The core implementation performs no action.

    * **Syntax:**
      ```python
      _on_llm_start(
          self,
          run: Run # Started LLM run
      ) -> Coroutine[
          Any,
          Any,
          None
      ] | None
      ```

30. `_on_llm_new_token`: Provides a protected hook called when an LLM or chat model produces a token or content blocks.

    The core implementation performs no action.

    * **Syntax:**
      ```python
      _on_llm_new_token(
          self,
          run: Run, # Run receiving the token
          token: str
          | list[
              str
              | dict[
                  str,
                  Any
              ]
          ], # Token or structured content blocks
          chunk: GenerationChunk
          | ChatGenerationChunk
          | None # Optional generation chunk
      ) -> Coroutine[
          Any,
          Any,
          None
      ] | None
      ```

31. `_on_llm_end`: Provides a protected hook called after successful LLM or chat-model completion.

    The core implementation performs no action.

    * **Syntax:**
      ```python
      _on_llm_end(
          self,
          run: Run # Completed model run
      ) -> Coroutine[
          Any,
          Any,
          None
      ] | None
      ```

32. `_on_llm_error`: Provides a protected hook called after an LLM or chat-model error.

    The core implementation performs no action.

    * **Syntax:**
      ```python
      _on_llm_error(
          self,
          run: Run # Errored model run
      ) -> Coroutine[
          Any,
          Any,
          None
      ] | None
      ```

33. `_on_chain_start`: Provides a protected hook called when a chain run starts.

    The core implementation performs no action.

    * **Syntax:**
      ```python
      _on_chain_start(
          self,
          run: Run # Started chain run
      ) -> Coroutine[
          Any,
          Any,
          None
      ] | None
      ```

34. `_on_chain_end`: Provides a protected hook called after successful chain completion.

    The core implementation performs no action.

    * **Syntax:**
      ```python
      _on_chain_end(
          self,
          run: Run # Completed chain run
      ) -> Coroutine[
          Any,
          Any,
          None
      ] | None
      ```

35. `_on_chain_error`: Provides a protected hook called after a chain error.

    The core implementation performs no action.

    * **Syntax:**
      ```python
      _on_chain_error(
          self,
          run: Run # Errored chain run
      ) -> Coroutine[
          Any,
          Any,
          None
      ] | None
      ```

36. `_on_tool_start`: Provides a protected hook called when a tool run starts.

    The core implementation performs no action.

    * **Syntax:**
      ```python
      _on_tool_start(
          self,
          run: Run # Started tool run
      ) -> Coroutine[
          Any,
          Any,
          None
      ] | None
      ```

37. `_on_tool_end`: Provides a protected hook called after successful tool completion.

    The core implementation performs no action.

    * **Syntax:**
      ```python
      _on_tool_end(
          self,
          run: Run # Completed tool run
      ) -> Coroutine[
          Any,
          Any,
          None
      ] | None
      ```

38. `_on_tool_error`: Provides a protected hook called after a tool error.

    The core implementation performs no action.

    * **Syntax:**
      ```python
      _on_tool_error(
          self,
          run: Run # Errored tool run
      ) -> Coroutine[
          Any,
          Any,
          None
      ] | None
      ```

39. `_on_chat_model_start`: Provides a protected hook called when a chat-model run starts.

    The core implementation performs no action.

    * **Syntax:**
      ```python
      _on_chat_model_start(
          self,
          run: Run # Started chat-model run
      ) -> Coroutine[
          Any,
          Any,
          None
      ] | None
      ```

40. `_on_retriever_start`: Provides a protected hook called when a retriever run starts.

    The core implementation performs no action.

    * **Syntax:**
      ```python
      _on_retriever_start(
          self,
          run: Run # Started retriever run
      ) -> Coroutine[
          Any,
          Any,
          None
      ] | None
      ```

41. `_on_retriever_end`: Provides a protected hook called after successful retriever completion.

    The core implementation performs no action.

    * **Syntax:**
      ```python
      _on_retriever_end(
          self,
          run: Run # Completed retriever run
      ) -> Coroutine[
          Any,
          Any,
          None
      ] | None
      ```

42. `_on_retriever_error`: Provides a protected hook called after a retriever error.

    The core implementation performs no action.

    * **Syntax:**
      ```python
      _on_retriever_error(
          self,
          run: Run # Errored retriever run
      ) -> Coroutine[
          Any,
          Any,
          None
      ] | None
      ```